#INIT

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Read Bronze Products

In [0]:
bronze_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/bronze/csv/products/products.csv"

df_products = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(bronze_path)
)

display(df_products)

In [0]:
df_products.printSchema()

print("Total records:", df_products.count())

#Check data quality

## check duplicate

In [0]:
df_products.groupBy("product_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

## check null

In [0]:
df_products.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_products.columns
]).show()

#Clean the Products data

In [0]:
df_products_clean = (
    df_products
    .dropDuplicates()
    .dropDuplicates(["product_id"])
    .filter(col("product_id").isNotNull())
    .withColumn("product_id", upper(trim(col("product_id"))))
    .withColumn("product_name", trim(col("product_name")))
    .withColumn("brand", trim(col("brand")))
    .withColumn("category", trim(col("category")))
    .withColumn("unit_price", col("unit_price").cast("decimal(10,2)"))
    .filter(col("unit_price") > 0)
)

In [0]:
display(df_products_clean)

#Validate the cleaned data

In [0]:
print("Bronze records :", df_products.count())
print("Silver records :", df_products_clean.count())

display(df_products_clean)

In [0]:
# schema
df_products_clean.printSchema()

# Write Silver as Delta

In [0]:
silver_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/silver/products/"
(
    df_products_clean.write
    .format("delta")
    .mode("overwrite")
    .save(silver_path)
)

# Verify Silver Delta

In [0]:
df_silver_products = (
    spark.read
    .format("delta")
    .load(silver_path)
)

display(df_silver_products)